# Block 3 — Nonverbal Marks & Verbal Handwriting
**Medical Document Intelligence System**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block3/block3/Block_3_Handwriting_Recognition.ipynb)

Block 3 has two **independent** recognizers, then a combined path that reads a **saved Block 1 ZIP** and imports the **Block 2 Knowledge Graph JSON** (not Block 2's per-sheet batch).

| Path | Engine | Input |
|---|---|---|
| Nonverbal | PaddleOCR (density fallback) | checkbox crops |
| Verbal | TrOCR | handwriting crops (tubes = digits; `others` raw) |
| Together | both + `KnowledgeGraph.load()` | `block1_normalized_batch.zip` + `kg/*.json` |

Qwen is **not** in this pass. Block 4 (KG constraints / rescoring) is later.

> **PHI:** Do not upload real clinic request sheets to Colab. This notebook uses the committed blank / synthetic sample. Clinic product later runs locally.

## 1. Setup
Clone the repo so `block1/`, `block2/`, and `block3/` are on disk, then import from `src/` (full package). Light dependencies only — Paddle and TrOCR are installed in their own cells.

In [ ]:
import os, sys, json, glob, shutil, zipfile

BRANCH = "block3"
REPO_URL = "https://github.com/RwaRwa599/epq3.git"

if "google.colab" in sys.modules or os.path.exists("/content"):
    repo_dir = "/content/repo"
    if os.path.exists(repo_dir):
        get_ipython().system(f"cd /content/repo && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}")
    else:
        get_ipython().system(f"git clone -b {BRANCH} {REPO_URL} /content/repo")
    src_path = "/content/repo/src"
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    os.chdir("/content/repo")
else:
    root = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "block3" else "."))
    src_path = os.path.join(root, "src")
    if os.path.isdir(src_path) and src_path not in sys.path:
        sys.path.insert(0, src_path)
    os.chdir(root)

for mod in list(sys.modules.keys()):
    if mod == "med_doc" or mod.startswith("med_doc."):
        del sys.modules[mod]

get_ipython().system('pip install -q "pydantic>=2.0.0" "numpy>=1.24.0" "opencv-python-headless>=4.8.0" "Pillow>=10.0.0" "matplotlib>=3.7.0" pandas')

from med_doc.htr.nonverbal import classify_marks, paddle_available
from med_doc.htr.verbal import recognize_fields, trocr_available
from med_doc.kg import KnowledgeGraph
print("✓ src/med_doc imported")
print("  PaddleOCR ready:", paddle_available(), " (install in the nonverbal cell)")
print("  TrOCR ready:   ", trocr_available(), " (install in the verbal cell)")
print("  Do not upload real clinic PHI to Colab.")

## 2. Nonverbal only (no TrOCR)
Install PaddleOCR and classify synthetic checkbox crops. This cell must not need `transformers`.

In [ ]:
import numpy as np
import pandas as pd
from med_doc.htr.nonverbal import classify_marks, paddle_available

get_ipython().system('pip install -q paddlepaddle "paddleocr>=2.7,<3"')

for mod in list(sys.modules.keys()):
    if mod.startswith("med_doc.htr.nonverbal") or mod.startswith("paddle"):
        del sys.modules[mod]
from med_doc.htr.nonverbal import classify_marks, paddle_available

def empty_box(n=48):
    img = np.full((n, n, 3), 245, dtype=np.uint8)
    img[0:2, :] = 20; img[-2:, :] = 20; img[:, 0:2] = 20; img[:, -2:] = 20
    return img

def tick_box(n=48):
    img = empty_box(n)
    for i in range(8, n - 8):
        img[i, i] = 15
        img[i, min(n - 1, i + 1)] = 15
    return img

crops = {"cbc": tick_box(), "alt": empty_box(), "ast": empty_box(), "profile_lipid": tick_box()}
marks = classify_marks(crops)
nv_table = pd.DataFrame([
    {"field_id": fid, "is_marked": p.is_marked, "confidence": p.confidence, "source": p.source, "ink_density": p.ink_density}
    for fid, p in marks.items()
])
print("PaddleOCR available:", paddle_available())
print("TrOCR was not imported for this cell. Source is paddle or density_fallback.")
display(nv_table)
assert marks["cbc"].is_marked is True
assert marks["alt"].is_marked is False
print("✓ Nonverbal path ran without TrOCR")

## 3. Verbal only (no Paddle)
Install TrOCR and transcribe handwriting crops. This cell must not need PaddleOCR.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from med_doc.htr.verbal import recognize_fields, trocr_available

get_ipython().system('pip install -q transformers torch')

def paper(w=240, h=64):
    return np.full((h, w, 3), 250, dtype=np.uint8)

def write_text(text, w=240, h=64):
    img = Image.fromarray(paper(w, h))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 36)
    except Exception:
        font = ImageFont.load_default()
    draw.text((12, 10), text, fill=(20, 20, 20), font=font)
    return np.asarray(img)

hw_crops = {
    "tube_edta": write_text("1"),
    "others": write_text("triglyc"),
    "clinical_info": paper(),
}
fields = recognize_fields(hw_crops, backend="trocr")
vb_table = pd.DataFrame([
    {"field_id": fid, "raw_text": p.raw_text, "confidence": p.confidence, "source": p.source, "needs_hitl": p.needs_hitl}
    for fid, p in fields.items()
])
print("TrOCR available:", trocr_available())
print("PaddleOCR was not used for this cell.")
display(vb_table)
print("✓ Verbal path ran without Paddle")

## 4. Together — Block 1 ZIP + Block 2 KG
Normalize the committed synthetic blank with Block 1, load the frozen Block 2 catalogue, run both recognizers, and print `hypotheses.json` confidences.

A saved `block1_normalized_batch.zip` also works (no need to keep Block 1 Colab open). Optional upload is offered; if you skip it, the sample is used.

In [ ]:
from pathlib import Path
import pandas as pd
from med_doc.normalization import normalize_batch
from med_doc.kg import KnowledgeGraph
from med_doc.htr import process_from_block1
from med_doc.paths import DEFAULT_KG, SYNTHETIC_DIR

input_zip = "block1_normalized_batch.zip"
UPLOAD_BLOCK1_ZIP = False  # set True only to pick a saved synthetic ZIP (never PHI)
if UPLOAD_BLOCK1_ZIP:
    try:
        from google.colab import files
        print("Upload a saved Block 1 ZIP (synthetic/blank only — no PHI).")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                input_zip = name
                break
    except Exception:
        print("Upload skipped.")
else:
    print("Using repo synthetic sample (set UPLOAD_BLOCK1_ZIP = True to pick a ZIP).")

if not os.path.exists(input_zip):
    samples = sorted(SYNTHETIC_DIR.glob("*.png"))
    if not samples:
        samples = sorted(Path("block1/samples").glob("*.png"))
    print(f"Running Block 1 on {len(samples)} synthetic sample(s)...")
    normalize_batch(samples, output_zip=input_zip)

kg = KnowledgeGraph.load(DEFAULT_KG)
print(f"Block 2 KG imported: version={kg.version}  catalogue={len(kg.catalogue)}")

result = process_from_block1(
    input_zip,
    output_zip="block3_predictions_batch.zip",
    kg=kg,
    backend="auto",
    mode="both",
)
manifest = result["manifest"]
print(f"Documents: {manifest['total_documents']}   HiTL: {manifest['hitl_documents']}")

out_dir = Path(result["output_dir"])
hyp_path = next(out_dir.glob("docs/*/hypotheses.json"))
hyp = json.loads(hyp_path.read_text())
print("hypotheses.json →", hyp_path)

nv = pd.DataFrame([
    {"path": "nonverbal", "field_id": fid, "is_marked": v.get("is_marked"), "raw_text": "", "confidence": v.get("confidence"), "source": v.get("source")}
    for fid, v in hyp["nonverbal"].items()
])
vb = pd.DataFrame([
    {"path": "verbal", "field_id": fid, "is_marked": None, "raw_text": v.get("raw_text"), "confidence": v.get("confidence"), "source": v.get("source")}
    for fid, v in hyp["verbal"].items()
])
print(f"ticked_test_ids ({len(hyp['ticked_test_ids'])}):", hyp["ticked_test_ids"][:20])
print("overall_confidence:", hyp["overall_confidence"])
print("\nNonverbal (ticked only, or first 12):")
ticked = nv[nv["is_marked"] == True]
display(ticked if len(ticked) else nv.head(12))
print("\nVerbal:")
display(vb)

try:
    from google.colab import files
    files.download("block3_predictions_batch.zip")
except Exception:
    print("ZIP saved at", os.path.abspath("block3_predictions_batch.zip"))